# CA-DVAE Sweep — Google Colab

**Before running:** Go to Runtime → Change runtime type → **T4 GPU**

**Runtime:** ~8-10 hours total. Both sweeps are resumable.

In [ ]:
# Clone repo and install
!git clone https://github.com/Vittal-Mukunda/Practicing-Agent-Looping.git /content/cadvae
%cd /content/cadvae
!pip install -e . --quiet

In [ ]:
# Verify GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Download datasets from Kaggle
# You need a Kaggle API token: kaggle.com → Account → Create New Token
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Upload your kaggle.json when prompted
from google.colab import files
uploaded = files.upload()  # upload kaggle.json
!mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

# Download datasets
!mkdir -p data/raw
!kaggle datasets download -d iakash17/customer-personality-analysis -p data/raw --unzip
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store -p data/raw -f 2019-Oct.csv --force

In [ ]:
# Unzip ecommerce data if needed
import os
if os.path.exists('data/raw/2019-Oct.csv.zip'):
    !cd data/raw && unzip -o 2019-Oct.csv.zip && rm 2019-Oct.csv.zip
!ls -lh data/raw/

In [ ]:
# Run Dataset A sweep (~2h)
!python -m cadvae.eval.run_cadvae_sweep data=personality

In [ ]:
# Run Dataset B sweep (~8h)
!python -m cadvae.eval.run_cadvae_sweep data=ecommerce eval.train_subsample=200000 model.max_epochs=40

In [ ]:
# Download results
!zip -r /content/phase5_results.zip results/phase5/
from google.colab import files
files.download('/content/phase5_results.zip')